# Tu copia personal · Aula UTP

Este cuaderno se abre desde GitHub. En Colab elige **Archivo → Guardar una copia en Drive** antes de conservar cambios. Renombra esa copia y ejecuta de arriba abajo. Cada estudiante trabaja en su propio archivo y entorno.

Los ejemplos son didácticos: sigue la sección indicada por Cristian para esta clase. Las comprobaciones son voluntarias, sin entregas ni notas.

# Lectura guiada · Machine Learning UTP
Cristian Alvarado · Ejecuta de arriba hacia abajo. No hay tareas ni entregas. Los ejemplos se revisan con el docente.

## Ruta de esta práctica · ML · Encuentro 4 · Aprender una ruta

**El puente con el aula:** el tablero es el mismo tipo de bodega imaginaria que recorriste a mano. Allí elegías movimientos para entender sus consecuencias. Aquí el programa realiza 2000 intentos y actualiza una regla para elegirlos. No es una aplicación de riego, un robot real ni un dron desplegado.

**Recorrido:** leer el mapa y los puntos elegidos → observar cómo se aprende durante muchos intentos → distinguir el mapa de flechas de la curva → seguir una demostración final sin seguir aprendiendo.

Guarda una copia personal en Drive. La celda principal contiene el ejemplo completo: ejecútala de arriba abajo. Usa CPU; no requiere claves de API, servicios ni recursos de pago. Si reinicias la sesión vuelve a ejecutar la celda. Las comprobaciones son voluntarias y sin nota.

## Q-learning: probar movimientos y aprender de sus consecuencias

**Situación:** una ficha debe ir desde S (inicio) hasta G (pedido/meta). Cada cuadrito libre es un lugar posible; # es una pared y X un pozo. GridWorld significa este mundo de casillas inventado. La ficha puede intentar subir, bajar, ir a la izquierda o a la derecha.

**Estado** es el lugar actual. **Acción** es el movimiento que intenta. **Entorno** es el tablero y sus reglas. **Recompensa** es el número que esas reglas devuelven después: −1 por un paso normal, +10 al entrar en G y −10 al entrar en X. Intentar cruzar una pared o el borde resta 1 aunque la ficha no se mueva. Los puntos fueron elegidos por nosotros; no son tiempo ni dinero observado.

Un **episodio** es un intento desde S hasta terminar o alcanzar el máximo de pasos. La **política** es la regla que elige el movimiento en cada casilla. Q-learning aprende números que comparan las consecuencias futuras de cada movimiento, en lugar de recibir una etiqueta de «movimiento correcto».

### Entiende qué se aprende antes de abrir la fórmula

Imagina una ficha por casilla con cuatro estimaciones: lo que podría conseguir si empieza subiendo, bajando, yendo a la izquierda o a la derecha. Esas estimaciones forman la **tabla Q**. Al principio todas valen 0; Q-learning las corrige después de cada movimiento.

El programa elige unas veces al azar para probar caminos (**explorar**) y otras aprovecha la mejor estimación disponible. **Épsilon (ε)** es la probabilidad de explorar: empieza en 1 (siempre probar al azar) y baja hasta 0,05. **Alfa (α=0,1)** es cuánto se corrige una estimación en cada paso. **Gamma (γ=0,95)** reduce el peso de puntos que llegarían más adelante. Estos tres valores se eligen; los números Q se aprenden.

La idea de una actualización es: estimación anterior + tamaño del ajuste × diferencia con el nuevo objetivo. En la fórmula, **s** es casilla actual, **a** acción, **r** recompensa recibida y «siguiente» la casilla a la que llegó. Si no terminó, el objetivo es r + γ × mejor Q del siguiente estado. Si terminó, el objetivo es solo r: no inventamos puntos después de la meta o del pozo.

**Un paso con valores dados:** Q anterior 2, recompensa −1, mejor Q siguiente 4. Objetivo = −1+0,95×4 = 2,8. Diferencia = 2,8−2 = 0,8. Corrección = 0,1×0,8 = 0,08. Q nuevo = **2,08**. Si ese paso hubiese terminado en meta con recompensa 10, el objetivo sería 10 y Q nuevo sería **2,8**. Esos Q de ejemplo explican la operación; no se fijan dentro del aprendizaje.

**Tip:** la curva suma puntos de cada intento sin descuento. La tabla Q estima puntos futuros con descuento. Aunque ambos hablen de recompensas, no son el mismo número.

In [ ]:
from IPython.display import display
# ======================================================================
# Q-LEARNING TABULAR (Colab listo para pegar)
# Caso didáctico: GridWorld 5x5 con paredes, meta (recompensa +10) y pozo (-10)
# ======================================================================
# Objetivo:
# - Aprender una política π(s) con alto retorno esperado descontado (gamma=0.95).
# - Q-learning estima Q[s, a] (valor de acción) iterando:
#   Q[s,a] ← Q[s,a] + α * (r + γ * max_a' Q[s',a'] - Q[s,a])
#
# Conceptos clave:
# - Tabla Q: matriz estados×acciones con valores estimados.
# - Política ε-greedy: con prob. ε explora; con 1-ε explota el mejor Q.
# - Aprendizaje fuera de política (off-policy): usa el max_a' Q[s',a'].
# ======================================================================

import numpy as np
np.random.seed(42)
import pandas as pd
import matplotlib.pyplot as plt

# --------------------------------------------------------
# 1) Definición del entorno GridWorld (discreto y tabular)
# --------------------------------------------------------
# Rejilla 5x5. Estados = celdas accesibles.
# Acciones: 0=Arriba, 1=Abajo, 2=Izquierda, 3=Derecha.
# Recompensas:
# - Paso normal: -1 (incentiva trayecto corto)
# - Meta (G): +10 y episodio termina
# - Pozo (X): -10 y episodio termina
# Paredes (#): no se puede entrar; quedarse en sitio penaliza -1
# Bordes: moverse fuera del tablero no está permitido; se ignora el movimiento y se penaliza -1.

H, W = 5, 5

# Mapa:
# S = inicio, G = meta, X = pozo, # = pared, . = libre
grid = np.array([
    list("S...."),
    list(".#..G"),
    list("..#.."),
    list("..X.."),
    list("....."),
])

start_pos = (0, 0)

# Utilidades de entorno
ACTIONS = [(-1,0), (1,0), (0,-1), (0,1)]  # U, D, L, R
N_ACTIONS = 4

def in_bounds(r, c):
    return 0 <= r < H and 0 <= c < W

def is_wall(r, c):
    return grid[r, c] == '#'

def to_state(r, c):
    return r * W + c

def from_state(s):
    return divmod(s, W)

def reset():
    return to_state(*start_pos)

def step(state, action):
    r, c = from_state(state)
    dr, dc = ACTIONS[action]
    nr, nc = r + dr, c + dc

    # Intento de movimiento
    if not in_bounds(nr, nc) or is_wall(nr, nc):
        nr, nc = r, c  # rebote
        reward = -1
        done = False
    else:
        cell = grid[nr, nc]
        if cell == 'G':
            reward = +10
            done = True
        elif cell == 'X':
            reward = -10
            done = True
        else:
            reward = -1
            done = False

    next_state = to_state(nr, nc)
    return next_state, reward, done

# Estados válidos (todas las celdas no pared)
mask_valid = (grid != '#')
N_STATES = H * W

# --------------------------------------------------------
# 2) Hiperparámetros de Q-learning
# --------------------------------------------------------
alpha = 0.1      # tasa de aprendizaje
gamma = 0.95     # factor de descuento
epsilon_start = 1.0
epsilon_end   = 0.05
epsilon_decay_episodes = 800  # episodios para decaer de start a end

episodes = 2000
max_steps = 200  # seguridad por episodio

# --------------------------------------------------------
# 3) Inicialización de la tabla Q
# --------------------------------------------------------
Q = np.zeros((N_STATES, N_ACTIONS), dtype=np.float32)

# Política ε-greedy
def epsilon_by_episode(ep):
    # Decaimiento lineal simple
    t = min(1.0, ep / epsilon_decay_episodes)
    return epsilon_start + t * (epsilon_end - epsilon_start)

def select_action(state, epsilon):
    if np.random.rand() < epsilon:
        return np.random.randint(N_ACTIONS)
    # Explota el mejor Q; desempata al azar
    qs = Q[state]
    max_q = np.max(qs)
    best_actions = np.flatnonzero(qs == max_q)
    return np.random.choice(best_actions)

# --------------------------------------------------------
# 4) Entrenamiento
# --------------------------------------------------------
returns = []

for ep in range(episodes):
    s = reset()
    total_reward = 0.0
    eps = epsilon_by_episode(ep)

    for t in range(max_steps):
        a = select_action(s, eps)
        s_next, r, done = step(s, a)

        # Regla central de Q-learning (off-policy)
        td_target = r + gamma * np.max(Q[s_next]) * (0 if done else 1)
        td_error  = td_target - Q[s, a]
        Q[s, a]  += alpha * td_error

        s = s_next
        total_reward += r
        if done:
            break

    returns.append(total_reward)

print("Entrenamiento completado.")
print("Recompensa media últimas 100 eps:", np.mean(returns[-100:]))

# --------------------------------------------------------
# 5) Visualización de aprendizaje
# --------------------------------------------------------
plt.figure(figsize=(7,3))
plt.plot(returns, linewidth=1)
plt.title("Suma de recompensas sin descuento por episodio exploratorio")
plt.xlabel("Episodio")
plt.ylabel("Retorno sin descuento")
plt.grid(True)
plt.show()

# --------------------------------------------------------
# 6) Extraer política greedy aprendida y mostrar tabla Q legible
# --------------------------------------------------------
# Política: acción con mayor Q por estado. Estados inválidos (#) se marcan.
arrow_map = {0:'^', 1:'v', 2:'<', 3:'>'}

policy_chars = grid.copy()
for r in range(H):
    for c in range(W):
        if grid[r,c] in ['#','G','X','S']:
            continue
        s = to_state(r,c)
        a = np.argmax(Q[s])
        policy_chars[r,c] = arrow_map[a]

print("Política greedy aprendida (flechas) sobre el mapa:")
for r in range(H):
    print(''.join(policy_chars[r]))

# Tabla Q como DataFrame (solo para estados no pared)
rows = []
for r in range(H):
    for c in range(W):
        if is_wall(r,c):
            continue
        s = to_state(r,c)
        rows.append({
            "estado": f"({r},{c})",
            "Q_up(^)":   round(float(Q[s,0]), 2),
            "Q_down(v)": round(float(Q[s,1]), 2),
            "Q_left(<)": round(float(Q[s,2]), 2),
            "Q_right(>)":round(float(Q[s,3]), 2),
            "celda": grid[r,c]
        })

q_df = pd.DataFrame(rows)
display(q_df.head(10))

# --------------------------------------------------------
# 7) Demostración: ejecutar un episodio con la política aprendida
# --------------------------------------------------------
def greedy_action(state):
    qs = Q[state]
    max_q = np.max(qs)
    best_actions = np.flatnonzero(qs == max_q)
    return np.random.choice(best_actions)

s = reset()
trajectory = [from_state(s)]
total = 0
for t in range(50):
    a = greedy_action(s)
    s, r, done = step(s, a)
    total += r
    trajectory.append(from_state(s))
    if done:
        break

print("Trayectoria con política aprendida:", trajectory)
print("Recompensa total de la demostración:", total)

# Visualización de la trayectoria
vis = grid.copy()
for (rr,cc) in trajectory:
    if vis[rr,cc] == '.':
        vis[rr,cc] = '*'
print("Mapa con trayectoria (*):")
for r in range(H):
    print(''.join(vis[r]))

# --------------------------------------------------------
# 8) Guía de lectura de resultados
# --------------------------------------------------------
print("""
================= CÓMO LEER LOS RESULTADOS =================
1) Curva de recompensa:
   - Muestra sumas sin descuento de episodios con exploración ε-greedy.
   - Una tendencia ascendente es evidencia del entrenamiento, no prueba de optimalidad.
   - Evaluar la política greedy por separado distingue aprendizaje de exploración.

2) Tabla Q:
   - Q[s,a] estima el retorno esperado descontado de esa acción y su continuación.
   - Cerca de un pozo, entrar en él tiene valor bajo; otra acción puede conducir a la meta.
   - El valor depende de la acción y de la ruta, no solo de la cercanía a una casilla.

3) Política greedy aprendida (flechas):
   - Flechas muestran el argmax de la tabla Q estimada en cada celda libre.
   - La demostración comprueba una ruta desde S; no prueba optimalidad en todo el mapa.

4) Demostración:
   - 'Trayectoria' recorre celdas desde inicio hasta terminar.
   - Si termina en 'G' con pocos pasos y buena recompensa, la política es efectiva.

5) Parámetros:
   - α (learning rate) controla cuánto ajustas Q en cada paso.
   - γ (descuento) reduce el peso de recompensas futuras en Q; el gráfico suma sin descuento.
   - ε permite probar acciones alternativas; decae para aprovechar más lo aprendido.
   - Un entrenamiento finito no garantiza haber aprendido la mejor acción en todos los estados.
=============================================================
""")


### Después de ejecutar: tres salidas, tres preguntas

**1. Curva de episodios:** el eje horizontal cuenta intentos; el vertical suma los puntos recibidos en cada intento. Un punto bajo puede incluir pasos exploratorios o caída al pozo. No dibuja el camino físico ni es un porcentaje de aciertos.

**2. Mapa de flechas y tabla Q:** cada flecha señala una acción con la estimación más alta en esa casilla; si hay empate puede elegirse otra equivalente. La tabla muestra cuatro estimaciones por estado. `(0,0)` es fila 0, columna 0: los índices empiezan en cero; no son coordenadas geográficas. Filas con G o X son terminales y no requieren continuar moviéndose.

**3. Trayectoria final:** la lista de casillas y el mapa con * permiten seguir una demostración. Mira dónde termina y cuenta **movimientos = número de casillas de la lista − 1**, porque la primera casilla es el inicio. Si llega a G en cinco movimientos, recibió cuatro penalizaciones de −1 y +10 al llegar: total **6**. La ruta puede ser otra; comprueba la salida obtenida y no inventes que fue óptima en todo el mapa.

**Prueba pequeña:** en tu copia repite solo el fragmento «Demostración» desde `s = reset()` hasta su impresión, manteniendo Q fija. No repitas el entrenamiento. Elige la mejor estimación disponible sin exploración; puede desempatar rutas. Comprueba llegada, movimientos y puntos. Si modificas paredes o recompensas, debes volver a entrenar: las estimaciones anteriores correspondían a otro entorno.